# Sujet 2 — Compréhension et valorisation des clients e-commerce

Ce notebook propose une démarche analytique complète pour **mieux comprendre les clients**, **révéler des segments**, **analyser les facteurs influents** et **proposer des décisions marketing concrètes** pour le sujet 2 du mini-projet.

Dataset conseillé dans l'énoncé : **Customer Segmentation Tutorial in Python**.  
Place le fichier `Mall_Customers.csv` dans le même dossier que ce notebook avant l’exécution.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


## 1. Chargement des données

In [ ]:
file_path = "Mall_Customers.csv"
df = pd.read_csv(file_path)

print("Dimensions du dataset :", df.shape)
df.head()


## 2. Compréhension initiale

In [ ]:
print("Informations générales :")
df.info()


In [ ]:
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

print("\nNombre de doublons :", df.duplicated().sum())


In [ ]:
df.describe(include="all").T

## 3. Nettoyage léger et renommage des colonnes

In [ ]:
data = df.copy()

data.columns = [c.strip().replace(" ", "_") for c in data.columns]

rename_map = {
    "Annual_Income_(k$)": "Annual_Income",
    "Spending_Score_(1-100)": "Spending_Score"
}
data = data.rename(columns=rename_map)

if "CustomerID" in data.columns:
    data = data.drop(columns=["CustomerID"])

print(data.columns.tolist())
data.head()


## 4. Analyse exploratoire des données

In [ ]:
print("Répartition par genre :")
print(data["Gender"].value_counts())


In [ ]:
plt.figure(figsize=(6,4))
data["Gender"].value_counts().plot(kind="bar")
plt.title("Répartition des clients par genre")
plt.xlabel("Genre")
plt.ylabel("Nombre de clients")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
num_cols = data.select_dtypes(include=np.number).columns.tolist()
print("Colonnes numériques :", num_cols)

data[num_cols].hist(figsize=(12, 4), bins=20)
plt.suptitle("Distribution des variables numériques")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(data["Annual_Income"], data["Spending_Score"], alpha=0.7)
plt.title("Revenu annuel vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(data["Age"], data["Spending_Score"], alpha=0.7)
plt.title("Âge vs Spending Score")
plt.xlabel("Âge")
plt.ylabel("Spending Score (1-100)")
plt.tight_layout()
plt.show()


In [ ]:
gender_stats = data.groupby("Gender")[["Age", "Annual_Income", "Spending_Score"]].mean().round(2)
gender_stats


## 5. Préparation des données pour la segmentation

In [ ]:
features = ["Age", "Annual_Income", "Spending_Score"]
X = data[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dimensions des données transformées :", X_scaled.shape)


## 6. Recherche du nombre optimal de clusters

In [ ]:
inertias = []
silhouette_scores = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(7,5))
plt.plot(list(k_values), inertias, marker="o")
plt.title("Méthode du coude (Elbow Method)")
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertie")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
plt.plot(list(k_values), silhouette_scores, marker="o")
plt.title("Silhouette Score selon k")
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

results_k = pd.DataFrame({
    "k": list(k_values),
    "Inertia": inertias,
    "Silhouette": silhouette_scores
})
results_k


> Choisis la valeur de **k** qui te semble la plus pertinente en observant le coude et le meilleur compromis sur le silhouette score.  
Dans ce notebook, on prend par défaut **k = 5**, souvent utilisé pour ce dataset.


In [ ]:
optimal_k = 5

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
data["Cluster"] = kmeans.fit_predict(X_scaled)

print("Taille de chaque cluster :")
print(data["Cluster"].value_counts().sort_index())
data.head()


## 7. Visualisation des segments

In [ ]:
plt.figure(figsize=(8,6))
for cluster in sorted(data["Cluster"].unique()):
    subset = data[data["Cluster"] == cluster]
    plt.scatter(subset["Annual_Income"], subset["Spending_Score"], label=f"Cluster {cluster}", alpha=0.7)

centers_scaled = kmeans.cluster_centers_
centers = scaler.inverse_transform(centers_scaled)
plt.scatter(centers[:, 1], centers[:, 2], marker="X", s=250, label="Centres")
plt.title("Segmentation clients : Revenu annuel vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
for cluster in sorted(data["Cluster"].unique()):
    subset = data[data["Cluster"] == cluster]
    plt.scatter(subset["Age"], subset["Spending_Score"], label=f"Cluster {cluster}", alpha=0.7)

plt.title("Segmentation clients : Âge vs Spending Score")
plt.xlabel("Âge")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Profil des clusters

In [ ]:
cluster_profile = data.groupby("Cluster")[["Age", "Annual_Income", "Spending_Score"]].mean().round(2)
cluster_profile["Count"] = data["Cluster"].value_counts().sort_index()
cluster_profile


In [ ]:
cluster_gender = pd.crosstab(data["Cluster"], data["Gender"], normalize="index") * 100
cluster_gender.round(2)


## 9. Interprétation métier automatique

In [ ]:
def describe_cluster(row):
    age = row["Age"]
    income = row["Annual_Income"]
    score = row["Spending_Score"]

    if income >= 70 and score >= 60:
        return "Clients premium à forte valeur : revenu élevé et forte propension à dépenser."
    elif income >= 70 and score < 40:
        return "Clients aisés mais peu dépensiers : potentiel de conversion via offres personnalisées."
    elif income < 45 and score >= 60:
        return "Clients à budget modéré mais très engagés : bons candidats pour promotions ciblées."
    elif age >= 45 and score < 50:
        return "Clients plus âgés et prudents : stratégie de fidélisation et recommandations adaptées."
    else:
        return "Segment intermédiaire : comportement équilibré, à suivre avec campagnes CRM segmentées."

interpretations = cluster_profile.copy()
interpretations["Interpretation"] = interpretations.apply(describe_cluster, axis=1)
interpretations


## 10. Recommandations marketing

In [ ]:
recommendations = pd.DataFrame({
    "Cluster": interpretations.index,
    "Action_marketing": [
        "Programme VIP, avant-premières, offres premium" if "premium" in text.lower()
        else "Campagnes d'upsell, bundles personnalisés" if "aisés" in text.lower()
        else "Promotions ciblées, programme de fidélité" if "budget modéré" in text.lower()
        else "Communication rassurante, offres simples et utiles" if "plus âgés" in text.lower()
        else "Segmentation fine, tests A/B et relances automatiques"
        for text in interpretations["Interpretation"]
    ]
})
recommendations


## 11. Conclusion

Ce notebook permet de :
- comprendre la structure des données clients,
- explorer les comportements d'achat,
- segmenter les clients avec **K-Means**,
- profiler chaque segment,
- proposer des **actions marketing concrètes**.

